## Modules for Data Collection

In [ ]:
import pandas as pd
import numpy as np
import re
import requests
import requests_cache
requests_cache.install_cache("STA_141B_Final_Project_Code")
import json
import lxml.html as lx
from itertools import chain
from bs4 import BeautifulSoup as bs

## The Numbers Dataset

In [ ]:
## Extracting all of the data

def extract_data(number):
    Name,Date,production_bug,Domestic_Gross,Worldwide_Gross=[],[],[],[],[]
    url="https://www.the-numbers.com/movie/budgets/all" #the inital url for the numbers website
    header_num = {"user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36"}
    #This part use web script to extract the imformation from the number websie
    response_num = requests.get(url, headers = header_num)
    html_num = lx.fromstring(response_num.text)
    Raw=[x.text_content() for x in html_num.xpath("//table//a")]
    Name.append(Raw[1::2]) #extracts the movie name column
    Date.append(Raw[0::2]) #extracts the release date column 
    Rawdata=[float(x.text_content().replace("\xa0$","").replace(",","")) for x in html_num.xpath("//td[@class = 'data']")]
    production_bug.append(Rawdata[1::4]) #append the name data into the empty column name
    Domestic_Gross.append(Rawdata[2::4]) #append the domestic gross column into the empty column Domestic_Gross
    Worldwide_Gross.append(Rawdata[3::4]) #append  
    for i in range(101,100*number+2,100):
        url="https://www.the-numbers.com/movie/budgets/all/"+str(i)
        response_num = requests.get(url, headers = header_num)
        html_num = lx.fromstring(response_num.text)
        Raw=[x.text_content() for x in html_num.xpath("//table//a")]
        Rawdata=[float(x.text_content().replace("\xa0$","").replace(",","")) for x in html_num.xpath("//td[@class = 'data']")] #this is the part that we forget to get
        Name.append(Raw[1::2])
        Date.append(Raw[0::2])
        production_bug.append(Rawdata[1::4])
        Domestic_Gross.append(Rawdata[2::4])
        Worldwide_Gross.append(Rawdata[3::4])
    Date1=list(chain.from_iterable(Date))
    Date2=[item.replace('Unknown', '') for item in Date1]
    Date3=pd.to_datetime(Date2)
    Name1=list(chain.from_iterable(Name))
    production_bug1=list(chain.from_iterable(production_bug))
    Domestic_Gross1=list(chain.from_iterable(Domestic_Gross))
    Worldwide_Gross1=list(chain.from_iterable(Worldwide_Gross))
    dic={"Release Date":Date3,"Movie":Name1,"Production Budget":production_bug1,
         "Domestic Gross":Domestic_Gross1,"Worldwide Gross":Worldwide_Gross1}
    dic_dataframe = pd.DataFrame(dic)
    dic_dataframe['Movie'] = dic_dataframe['Movie'].astype(str) #convert Movie column to string
    dic_dataframe['Production Budget'] = dic_dataframe['Production Budget'].astype(float) #convert Movie column to float
    dic_dataframe['Domestic Gross'] = dic_dataframe['Domestic Gross'].astype(float) #convert Movie column to float
    dic_dataframe['Worldwide Gross'] = dic_dataframe['Worldwide Gross'].astype(float) #convert Movie column to string
    return dic_dataframe
final6340=extract_data(63)
final6340["Release Date"]=pd.DatetimeIndex(final6340['Release Date']).year
#i just added the raw data into the forloop, so the raw date will update for every i, and its working

In [ ]:
## Removing special characters

def unicodetoascii(text):
    TEXT = (text.
            replace('\\xe2\\x80\\x99', "'").
            replace('\\xc3\\xa9', 'e').
            replace('\\xe2\\x80\\x90', '-').
            replace('\\xe2\\x80\\x91', '-').
            replace('\\xe2\\x80\\x92', '-').
            replace('\\xe2\\x80\\x93', '-').
            replace('\\xe2\\x80\\x94', '-').
            replace('\\xe2\\x80\\x94', '-').
            replace('\\xe2\\x80\\x98', "'").
            replace('\\xe2\\x80\\x9b', "'").
            replace('\\xe2\\x80\\x9c', '"').
            replace('\\xe2\\x80\\x9c', '"').
            replace('\\xe2\\x80\\x9d', '"').
            replace('\\xe2\\x80\\x9e', '"').
            replace('\\xe2\\x80\\x9f', '"').
            replace('\\xe2\\x80\\xa6', '...').
            replace('\\xe2\\x80\\xb2', "'").
            replace('\\xe2\\x80\\xb3', "'").
            replace('\\xe2\\x80\\xb4', "'").
            replace('\\xe2\\x80\\xb5', "'").
            replace('\\xe2\\x80\\xb6', "'").
            replace('\\xe2\\x80\\xb7', "'").
            replace('\\xe2\\x81\\xba', "+").
            replace('\\xe2\\x81\\xbb', "-").
            replace('\\xe2\\x81\\xbc', "=").
            replace('\\xe2\\x81\\xbd', "(").
            replace('\\xe2\\x81\\xbe', ")").
            replace('â\x80\x99',"'").
            replace('â\x80¦'," ").
            replace("(ë°\x80ì\xa0\x95)","").
            replace("Ã©","é").
            replace("â\x80\x94"," ").
            replace("Ã´","ô").
            replace("Ã¼","ü").
            replace("Ã¯","i").
            replace("Ã","É").
            replace("É¨","è").
            replace("â\x80\x93","–").
            replace("É","à").
            replace("§","ç").
            replace("Âº","º").
            replace("à³","ó").
            replace("Â","").
            replace("à«","ë").
            replace("à¤","ä").
            replace("à»","û").
            replace("à","É").
            replace("Ép","Ép").
            replace("Éª","ê").
            replace("É£","ã").
            replace("É¥","å").
            replace("Éº","ú").
            replace("É¸","ø").
            replace("É","í"))
    return TEXT
final6340["Movie"]=[unicodetoascii(x) for x in final6340["Movie"]]
final6340[final6340["Domestic Gross"]==0] #no zero
finalno0=final6340[(final6340['Domestic Gross'] !=0) & (final6340['Worldwide Gross'] !=0) & (final6340['Production Budget'] !=0)].reset_index(drop=True)
finalno=finalno0[finalno0["Release Date"].notnull()].reset_index(drop=True)
finalno["Release Date"]=finalno["Release Date"].astype(np.int64) #convert time

In [ ]:
## Standardizing movie titles

def rare_title_list(list_title_upp, add_apos = True):
    common_character_list = [
    " ",
    "0","1","2","3","4","5","6","7","8","9",
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z"
    ]
    if add_apos == True:
        common_character_list.append("'")
    rare_title_list = []
    for title in list_title_upp:
        condition_title = []
        for character in title:
            condition_title.append(character not in common_character_list)
        if sum(condition_title) >= 1:
            rare_title_list.append(title)
    return rare_title_list
movie_tit = [x.upper() #this list keep tracks of the edits for the movie title because we want to standardize the movie title
             .replace(" : "," ")#
             .replace(": "," ")#
             .replace(" :"," ")#
             .replace(":"," ")#
             .replace(" . "," ")#
             .replace(". "," ")#
             .replace(" ."," ")#
             .replace("."," ")#
             .replace(" & "," AND ")#
             .replace("& "," AND ")#
             .replace(" &"," AND ")#
             .replace("&"," AND ")#
             .replace(" - "," ")# 
             .replace(" -"," ")#
             .replace("- "," ")#
             .replace("-"," ")#
             .replace("É","E")
             .replace(" \\ ","")#
             .replace("\\ ","")#
             .replace(" \\","")#
             .replace("\\","")#
             .replace("   "," ")#
             .replace(" , "," ")#
             .replace(" ,"," ")#
             .replace(", "," ")#
             .replace(","," ")#
             .replace(" ! "," ")#
             .replace(" !"," ")#
             .replace("! "," ")#
             .replace("!"," ")#
             .replace(" ? "," ")#
             .replace(" ?"," ")#
             .replace("? "," ")#
             .replace("?"," ")#
             .replace("Ü","U")
             .replace("Ë","E")
             .replace("È","E")
             .replace("É","E")
             .replace("È","E")
             .replace("Ä","A")
             .replace("Á","A")
             .replace("À","A")
             .replace("Í","I")
             .replace("Ç","C")
             .replace("Ô","O")
             .replace(" / "," ")#
             .replace(" /"," ")#
             .replace("/ "," ")#
             .replace("/"," ")#
             .replace("¢","C")
             .replace(" ( "," ")#
             .replace("( "," ")#
             .replace(" ("," ")#
             .replace("("," ")#
             .replace(" ) "," ")#
             .replace(" )"," ")#
             .replace(") "," ")#
             .replace(")"," ")#
             .replace(" [ "," ")#
             .replace("[ "," ")#
             .replace(" ["," ")#
             .replace("["," ")#
             .replace(" ] "," ")#
             .replace(" ]"," ")#
             .replace("] "," ")#
             .replace("]"," ")#
             .replace(" * "," ")
             .replace(" *"," ")
             .replace("* "," ")
             .replace("*"," ")
             .replace(" + "," AND ")
             .replace(" +"," AND ")
             .replace("+ "," AND ")
             .replace("+"," AND ")
             .replace(" ¦ "," ")
             .replace("¦ "," ")
             .replace(" ¦"," ")
             .replace("¦"," ")
             .replace(" # "," ")
             .replace(" #"," ")
             .replace("# "," ")
             .replace("#"," ")
             .replace(" º "," ")
             .replace(" º"," ")
             .replace("º "," ")
             .replace(" º "," ")
             .replace(" ¹ "," ")
             .replace("¹ "," ")
             .replace(" ¹"," ")
             .replace("¹"," ")
             .replace('THE CHRONICLES OF NARNIA THE LION THE WITCH A…','THE CHRONICLES OF NARNIA THE LION THE WITCH AND THE WARDROBE') 
             .replace('THE CHRONICLES OF NARNIA THE VOYAGE OF THE DAW…','THE CHRONICLES OF NARNIA THE VOYAGE OF THE DAWN TREADER')
             .replace('PIRATES OF THE CARIBBEAN THE CURSE OF THE BLAC…','PIRATES OF THE CARIBBEAN THE CURSE OF THE BLACK PEARL')
             .replace('BIRDS OF PREY AND THE FANTABULOUS EMANCIPATION…','BIRDS OF PREY AND THE FANTABULOUS EMANCIPATION OF ONE HARLEY QUINN')
             .replace('THE ASSASSINATION OF JESSE JAMES BY THE COWARD …','THE ASSASSINATION OF JESSE JAMES BY THE COWARD ROBERT FORD')
             .replace('ALEXANDER AND THE TERRIBLE HORRIBLE NO GOOD …','ALEXANDER AND THE TERRIBLE HORRIBLE NO GOOD VERY BAD DAY')
             .replace('TEENAGE MUTANT NINJA TURTLES II THE SECRET OF …','TEENAGE MUTANT NINJA TURTLES II THE SECRET OF THE OOZE')
             .replace("THE PIRATES WHO DON'T DO ANYTHING A VEGGIETALE…","THE PIRATES WHO DON'T DO ANYTHING A VEGGIETALES MOVIE")
             .replace('HANNAH MONTANA AND MILEY CYRUS BEST OF BOTH WO…','HANNAH MONTANA AND MILEY CYRUS BEST OF BOTH WORLDS CONCERT')
             .replace("HILLARY'S AMERICA THE SECRET HISTORY OF THE DE…","STREAME HILLARY'S AMERICA THE SECRET HISTORY OF THE DEMOCRATIC PARTY JETZT BEI DIESEN ANBIETERN")
             .replace("A NIGHTMARE ON ELM STREET PART 2 FREDDY'S REVE…","A NIGHTMARE ON ELM STREET PART 2 FREDDY'S REVE")
             .replace("MARILYN HOTCHKISS' BALLROOM DANCING AND CHARM S…","MARILYN HOTCHKISS' BALLROOM DANCING AND CHARM SCHOOL")
             .replace('POM WONDERFUL PRESENTS THE GREATEST MOVIE EVER…',"POM WONDERFUL PRESENTS THE GREATEST MOVIE EVER SOLD")
             .replace('ONCE IN A LIFETIME THE EXTRAORDINARY STORY OF …','ONCE IN A LIFETIME THE EXTRAORDINARY STORY OF THE NEW YORK COSMOS')
             .replace('AQUA TEEN HUNGER FORCE COLON MOVIE FILM FOR THE…','AQUA TEEN HUNGER FORCE COLON MOVIE FILM FOR THEATERS')
             .replace('DECEPTIVE PRACTICE THE MYSTERIES AND MENTORS O…','DECEPTIVE PRACTICE THE MYSTERIES AND MENTORS OF RICKY JAY')
             .replace('NÃO PARE NA PISTA A MELHOR HISTORIA DE PAULO C…',"NAO PARE NA PISTA A MELHOR HISTORIA DE PAULO COEL")
             .replace('CI¬KE NIE YINNIIING','CI KE NIE YIN NIANG')
             .replace('Ó',"O")
             .replace('¡',"I")
             .replace('Ú','U')
             .replace('Ê',"E")
             .replace('I\xad',"I")
             .replace('Å',"A")
             .replace('Û',"U")
             .replace('I\xa0',"A")
            for x in finalno["Movie"]]
removie_list = rare_title_list(movie_tit)
#this removie list shows the movies we want to filter out

In [ ]:
movie_tit = [x.rstrip(" ") if x.endswith(" ") else x for x in movie_tit]
movie_tit = [x.lstrip(" ") if x.startswith(" ") else x for x in movie_tit]
clean_movie_tit = pd.DataFrame(movie_tit) #clean movie title
tem_dataset = pd.concat([finalno,clean_movie_tit],axis=1)
tem_dataset.columns = [*tem_dataset.columns[:-1], 'Clean Movie Name']
desire_list = tem_dataset.Movie.isin(removie_list)
tem_dataset = pd.concat([finalno,clean_movie_tit],axis=1)
tem_dataset.columns = [*tem_dataset.columns[:-1], 'Clean Movie Name']
tem_dataset = tem_dataset.loc[[False if x in removie_list else True for x in tem_dataset["Clean Movie Name"]],:].reset_index(drop=True)
tem_dataset = tem_dataset.sort_values(by = "Clean Movie Name")
tem_dataset["Movie"] = [x + " (" + str(y) + ")" for x,y in zip(tem_dataset["Clean Movie Name"],tem_dataset["Release Date"])]
tem_dataset = tem_dataset.iloc[:,[1,2,3,4]]
tem_dataset = tem_dataset.reset_index(drop=True)

In [ ]:
tem_dataset

## IMDB Dataset

In [ ]:
def meta_score(i):
    return i.text_content().replace("\n","").replace("\xa0","").replace("                                                                        ","").replace("                                ","").replace("                    ","").replace("      ","").replace("    Metascore    ","")[-2:]

In [ ]:
## Top 1000 movies

first_half = 'https://www.imdb.com/search/title/?groups=top_1000&sort=boxoffice_gross_us,desc&start='
last_half = "&ref_=adv_nxt"

title = []
genre = []
movie_rating = []
user_rating = []
metascore = []
duration = []
director = []
lead = []
year = []

titlefinal = []
genrefinal = []
moviefinal = []
userfinal = []
metafinal = []
durationfinal = []
directorfinal = []
leadfinal = []
yearfinal = []

TitleData = pd.DataFrame()
GenreData= pd.DataFrame()
MovieData = pd.DataFrame()
UserData = pd.DataFrame()
MetaData = pd.DataFrame()
DurationData = pd.DataFrame()
DirectorData = pd.DataFrame()
LeadData = pd.DataFrame()
YearData = pd.DataFrame()

for i in range(1,1001, 50):  
    website = first_half + str(i) + last_half
    response_imdb = requests.get(website)
    html_imdb = lx.fromstring(response_imdb.text)
    #For lead and director
    soup = bs(response_imdb.text, 'html.parser')
    for i in range(0,len(soup.find_all("p", class_ = ""))):
        director_stars = soup.find_all("p", class_ = "")[i].get_text().replace("\n","").replace("    ","").replace("| ","").replace("Director:","").replace("Directors:","").replace("Stars:","*")
        director.append(director_stars.split("*")[0].split(", "))
        lead.append(director_stars.split("*")[1].split(", ")[0])
    
    title_page_i = [x.text_content() for x in html_imdb.xpath("//h3//a")]
    year_page_i = [x.text_content().replace('(I) ','') for x in html_imdb.xpath("//span[@class = 'lister-item-year text-muted unbold']")]
    #title_year_page_i = [x + " " + y for x,y in zip(title_page_i,year_page_i)]
    year.append(year_page_i)
    title.append(title_page_i)
    genre.append([x.text_content().replace("\n","").replace("            ","") for x in html_imdb.xpath("//span[@class = 'genre']")])
    #movie_rating.append([x.text_content() for x in html_imdb.xpath("//span[@class = 'certificate']")])
    movie_rating.append([x.text_content().replace("\n            ","").replace("     |     ","*").split("*")[0] if x.text_content().replace("\n            ","")[0] != " " else None for x in html_imdb.xpath("//p[@class = 'text-muted ']")])
    user_rating.append([float(x) for x in html_imdb.xpath("//div[@name = 'ir']//@data-value")])
    list_meta_score = [meta_score(x) if not ("X" in meta_score(x)) else None for x in html_imdb.xpath("//div[@class = 'ratings-bar']")]
    metascore.append([float(x) if x != None else x for x in list_meta_score])
    duration.append([float(x.text_content().replace("\n","").replace("            ","").replace(" min","")) for x in html_imdb.xpath("//span[@class = 'runtime']")])
            
for i in range(20):#create a for loop and i in range 20
    titlefinal += title[i]
    genrefinal += genre[i]
    moviefinal += movie_rating[i]
    userfinal += user_rating[i]
    metafinal += metascore[i]
    durationfinal += duration[i]
    yearfinal += year[i]

TitleData['Title'] = titlefinal
GenreData['Genre'] = genrefinal
MovieData['Movie Rating'] = moviefinal
UserData['User Rating'] = userfinal
MetaData['Meta Score'] = metafinal
DurationData['Duration in mins'] = durationfinal
DirectorData['Director'] = director
LeadData['Lead'] = lead
YearData['Year'] = yearfinal

combineTop = pd.concat([TitleData,GenreData,MovieData, UserData, MetaData, DurationData, DirectorData, LeadData,YearData], axis = 1)

In [ ]:
# Bottom 1000 movies

first_half = 'https://www.imdb.com/search/title/?groups=bottom_1000&sort=boxoffice_gross_us,desc&start='
last_half = "&ref_=adv_prv"

title = []
genre = []
movie_rating = []
user_rating = []
metascore = []
duration = []
director = []
lead = []
year = []

titlefinal = []
genrefinal = []
moviefinal = []
userfinal = []
metafinal = []
durationfinal = []
directorfinal = []
leadfinal = []
yearfinal = []

TitleData = pd.DataFrame()
GenreData= pd.DataFrame()
MovieData = pd.DataFrame()
UserData = pd.DataFrame()
MetaData = pd.DataFrame()
DurationData = pd.DataFrame()
DirectorData = pd.DataFrame()
LeadData = pd.DataFrame()
YearData = pd.DataFrame()

for i in range(1,1001, 50):  
    website = first_half + str(i) + last_half
    response_imdb = requests.get(website)
    html_imdb = lx.fromstring(response_imdb.text)
    #For lead and director
    soup = bs(response_imdb.text, 'html.parser')
    for i in range(0,len(soup.find_all("p", class_ = ""))):
        director_stars = soup.find_all("p", class_ = "")[i].get_text().replace("\n","").replace("    ","").replace("| ","").replace("Director:","").replace("Directors:","").replace("Stars:","*")
        director.append(director_stars.split("*")[0].split(", "))
        lead.append(director_stars.split("*")[1].split(", ")[0])
    
    title_page_i = [x.text_content() for x in html_imdb.xpath("//h3//a")]
    year_page_i = [x.text_content().replace('(I) ','') for x in html_imdb.xpath("//span[@class = 'lister-item-year text-muted unbold']")]
    #title_year_page_i = [x + " " + y for x,y in zip(title_page_i,year_page_i)]
    year.append(year_page_i)
    title.append(title_page_i)
    genre.append([x.text_content().replace("\n","").replace("            ","") for x in html_imdb.xpath("//span[@class = 'genre']")])
    #movie_rating.append([x.text_content() for x in html_imdb.xpath("//span[@class = 'certificate']")])
    movie_rating.append([x.text_content().replace("\n            ","").replace("     |     ","*").split("*")[0] if x.text_content().replace("\n            ","")[0] != " " else None for x in html_imdb.xpath("//p[@class = 'text-muted ']")])
    user_rating.append([float(x) for x in html_imdb.xpath("//div[@name = 'ir']//@data-value")])
    list_meta_score = [meta_score(x) if not ("X" in meta_score(x)) else None for x in html_imdb.xpath("//div[@class = 'ratings-bar']")]
    metascore.append([float(x) if x != None else x for x in list_meta_score])
    duration.append([float(x.text_content().replace("\n","").replace("            ","").replace(" min","")) for x in html_imdb.xpath("//span[@class = 'runtime']")])
            
for i in range(20):#create a for loop and i in range 20
    titlefinal += title[i]
    genrefinal += genre[i]
    moviefinal += movie_rating[i]
    userfinal += user_rating[i]
    metafinal += metascore[i]
    durationfinal += duration[i]
    yearfinal += year[i]

TitleData['Title'] = titlefinal
GenreData['Genre'] = genrefinal
MovieData['Movie Rating'] = moviefinal
UserData['User Rating'] = userfinal
MetaData['Meta Score'] = metafinal
DurationData['Duration in mins'] = durationfinal
DirectorData['Director'] = director
LeadData['Lead'] = lead
YearData['Year'] = yearfinal

combineBottom = pd.concat([TitleData,GenreData,MovieData, UserData, MetaData, DurationData, DirectorData, LeadData, YearData], axis = 1)

In [ ]:
# Combine Top 1000 and Bottom 1000 movies

FinalCombine = pd.concat([combineTop, combineBottom], axis = 0).reset_index(drop = True)
Userlist = []
for x in FinalCombine['User Rating']:
    if x >= 7:
        Userlist.append('1')
    elif x < 7:
        Userlist.append('0')
    else:
        Userlist.append(None)
Metalist = []
for x in FinalCombine['Meta Score']:
    if x >= 7:
        Metalist.append('1')
    elif x < 7:
        Metalist.append('0')
    else:
        Metalist.append(None)
FinalCombine['User Rating Good/Bad']=Userlist
FinalCombine['Meta Score Good/Bad']=Metalist
FinalCombine_No_NaN = FinalCombine.dropna().reset_index(drop=True) # Final dataset without NaN, which also drops any row with None
DirectorOne = []
for x in FinalCombine_No_NaN['Director']:
    if len(x) == 1:
        DirectorOne.append(1)
    else:
        DirectorOne.append(0)
FinalCombine_No_NaN['Director Count 0/1'] = DirectorOne
FinalCombine_No_NaN['Director Count'] = [len(x) for x in FinalCombine_No_NaN['Director']]

In [ ]:
def title_genre(desired_genre):
    return FinalCombine_No_NaN.loc[[True if x == desired_genre else False for x in FinalCombine_No_NaN["Genre"]],:]

## ISSUE BELOW

The code below is not runable because indices of observations we classified to be a particular genre for our project will not be the same indices when making a new request (IMDB updates its site with new movies)

In [ ]:
## Horror

FinalCombine_No_NaN['Genre'].value_counts().keys() #genres
n = FinalCombine_No_NaN['Genre'].value_counts() #freq
DataGenre= pd.DataFrame(n)
DataGenre = DataGenre.reset_index()
DataGenre.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq = DataGenre.loc[[True if "Horror" in x else False for x in DataGenre["Genre names"]],:]
list_to_horror = [df_genre_freq["Genre names"][index] for index in 
 [5,7,26,36,49,52,62,63,77,92,94,97,99,100,101,118,130,135,139,140,163,186,192,196,206]]
FinalCombine_No_NaN_to_horror = FinalCombine_No_NaN.copy()
FinalCombine_No_NaN_to_horror["Genre"] = ["Horror" if x in list_to_horror else x for x in FinalCombine_No_NaN["Genre"]]

In [ ]:
## Comedy

n1 = FinalCombine_No_NaN_to_horror['Genre'].value_counts() #freq
DataGenre1 = pd.DataFrame(n1)
DataGenre1 = DataGenre1.reset_index()
DataGenre1.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq1 = DataGenre1.loc[[True if "Comedy" in x else False for x in DataGenre1["Genre names"]],:]#.reset_index(drop = True)
list_to_h_comedy = [df_genre_freq1["Genre names"][index] for index in 
 [3,19,21,27,31,45,62,73,74,79,80,85,101,104,114,115,118,120,126,134,140,141,155,156,162,164,165,172,173,190,191,193,201,211,223,225,226,228]]
FinalCombine_No_NaN_to_h_comedy = FinalCombine_No_NaN_to_horror.copy()
FinalCombine_No_NaN_to_h_comedy["Genre"] = ["Comedy" if x in list_to_h_comedy else x for x in FinalCombine_No_NaN_to_horror["Genre"]]

In [ ]:
## Animation

n2 = FinalCombine_No_NaN_to_h_comedy['Genre'].value_counts() #freq
DataGenre2 = pd.DataFrame(n2)
DataGenre2 = DataGenre2.reset_index()
DataGenre2.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq2 = DataGenre2.loc[[True if "Animation" in x else False for x in DataGenre2["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_ani = [df_genre_freq2["Genre names"][index] for index in 
 [3,21,48,54,84,89,95,111,119,145,162,176,180,186,188,189,190]]
FinalCombine_No_NaN_to_h_c_ani = FinalCombine_No_NaN_to_h_comedy.copy()
FinalCombine_No_NaN_to_h_c_ani["Genre"] = ["Animation" if x in list_to_h_c_ani else x for x in FinalCombine_No_NaN_to_h_comedy["Genre"]]

In [ ]:
## Drama

n3 = FinalCombine_No_NaN_to_h_c_ani['Genre'].value_counts() #freq
DataGenre3 = pd.DataFrame(n3)
DataGenre3 = DataGenre3.reset_index()
DataGenre3.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq3 = DataGenre3.loc[[True if "Drama" in x else False for x in DataGenre3["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d = [df_genre_freq3["Genre names"][index] for index in 
 [3,12,13,14,16,19,21,25,33,35,44,46,52,57,59,60,61,67,68,72,82,93,100,105,107,108,109,117,123,135,148,164,167,172,174]]
FinalCombine_No_NaN_to_h_c_a_d = FinalCombine_No_NaN_to_h_c_ani.copy()
FinalCombine_No_NaN_to_h_c_a_d["Genre"] = ["Drama" if x in list_to_h_c_a_d else x for x in FinalCombine_No_NaN_to_h_c_ani["Genre"]]

In [ ]:
## Romance

n4 = FinalCombine_No_NaN_to_h_c_a_d['Genre'].value_counts() #freq
DataGenre4 = pd.DataFrame(n4)
DataGenre4 = DataGenre4.reset_index()
DataGenre4.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq4 = DataGenre4.loc[[True if "Romance" in x else False for x in DataGenre4["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r = [df_genre_freq4["Genre names"][index] for index in 
 [4,6,8,23,25,34,35,39,42,43,52,62,65,68,70,78,80,82,92,94,101,102,107,112,115,120,126,127,130,136]]
FinalCombine_No_NaN_to_h_c_a_d_r = FinalCombine_No_NaN_to_h_c_a_d.copy()
FinalCombine_No_NaN_to_h_c_a_d_r["Genre"] = ["Romance" if x in list_to_h_c_a_d_r else x for x in FinalCombine_No_NaN_to_h_c_a_d["Genre"]]

In [ ]:
## Biography

n5 = FinalCombine_No_NaN_to_h_c_a_d_r['Genre'].value_counts() #freq
DataGenre5 = pd.DataFrame(n5)
DataGenre5 = DataGenre5.reset_index()
DataGenre5.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq5 = DataGenre5.loc[[True if "Biography" in x else False for x in DataGenre5["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab = [df_genre_freq5["Genre names"][index] for index in 
 [12,15,17,32,35,44,52,84,88]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab = FinalCombine_No_NaN_to_h_c_a_d_r.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab["Genre"] = ["Biography" if x in list_to_h_c_a_d_r_ab else x for x in FinalCombine_No_NaN_to_h_c_a_d_r["Genre"]]

In [ ]:
## Sci-Fi/Fantasy

n6 = FinalCombine_No_NaN_to_h_c_a_d_r_ab['Genre'].value_counts() #freq
DataGenre6 = pd.DataFrame(n6)
DataGenre6 = DataGenre6.reset_index()
DataGenre6.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq6 = DataGenre6.loc[[True if "Sci-Fi" in x else False for x in DataGenre6["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sf = [df_genre_freq6["Genre names"][index] for index in 
 [19,22,25,27,35,40,51,65,68,72,73,76,77,88]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sf = FinalCombine_No_NaN_to_h_c_a_d_r_ab.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sf["Genre"] = ["Sci-fi/Fantasy" if x in list_to_h_c_a_d_r_ab_sf else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab["Genre"]]
#Fantasy
n7 = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sf['Genre'].value_counts() #freq
DataGenre7 = pd.DataFrame(n7)
DataGenre7 = DataGenre7.reset_index()
DataGenre7.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq7 = DataGenre7.loc[[True if "Fantasy" in x else False for x in DataGenre7["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sff = [df_genre_freq7["Genre names"][index] for index in 
 [6,49,65,81,85,87]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sf.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff["Genre"] = ["Sci-fi/Fantasy" if x in list_to_h_c_a_d_r_ab_sff else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab_sf["Genre"]]

In [ ]:
## Action/Adventure

n8 = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff['Genre'].value_counts() #freq
DataGenre8 = pd.DataFrame(n8)
DataGenre8 = DataGenre8.reset_index()
DataGenre8.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq8 = DataGenre8.loc[[True if "Action" in x else False for x in DataGenre8["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sff_a = [df_genre_freq8["Genre names"][index] for index in 
 [7,8,9,12,13,15,17,18,20,21,24,30,32,35,36,39,40,41,43,45,48,49,55,56,61,62,63,66,67,68,69,71,72,75,83]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_a = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_a["Genre"] = ["Action/Adventure" if x in list_to_h_c_a_d_r_ab_sff_a else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff["Genre"]]
n9 = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_a['Genre'].value_counts() #freq
DataGenre9 = pd.DataFrame(n9)
DataGenre9 = DataGenre9.reset_index()
DataGenre9.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq9 = DataGenre9.loc[[True if "Adventure" in x else False for x in DataGenre9["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sff_aa = [df_genre_freq9["Genre names"][index] for index in 
 [1,12,17,19,22,27,28,29,30,31,34,35,42,43,50,52]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_a.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa["Genre"] = ["Action/Adventure" if x in list_to_h_c_a_d_r_ab_sff_aa else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_a["Genre"]]

In [ ]:
## Thriller (included into Horror)

DataGenre10 = pd.DataFrame(FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa['Genre'].value_counts())
DataGenre10 = DataGenre10.reset_index()
DataGenre10.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq10 = DataGenre10.loc[[True if "Thriller" in x else False for x in DataGenre10["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sff_aa_t = [df_genre_freq10["Genre names"][index] for index in 
 [8,15,18,21,24,26,27,31,35,36,37]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t["Genre"] = ["Horror" if x in list_to_h_c_a_d_r_ab_sff_aa_t else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa["Genre"]]

In [ ]:
## Comedy-Drama

DataGenre11 = pd.DataFrame(FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t['Genre'].value_counts())
DataGenre11 = DataGenre11.reset_index()
DataGenre11.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq11 = DataGenre11.loc[[True if "Drama" in x else False for x in DataGenre11["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sff_aa_t_d = [df_genre_freq11["Genre names"][index] for index in 
 [8,10,11,12,13,14,15,16,18,21,22,23,24,25,26]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_d = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_d["Genre"] = ["Comedy-Drama" if x in list_to_h_c_a_d_r_ab_sff_aa_t_d else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t["Genre"]]
DataGenre12 = pd.DataFrame(FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_d['Genre'].value_counts())
DataGenre12 = DataGenre12.reset_index()
DataGenre12.rename(columns = {'index':'Genre names', 'Genre':'Frequency'}, inplace = True)
df_genre_freq12 = DataGenre12.loc[[True if "Comedy" in x else False for x in DataGenre12["Genre names"]],:]#.reset_index(drop = True)
list_to_h_c_a_d_r_ab_sff_aa_t_dc = [df_genre_freq12["Genre names"][index] for index in 
 [6,9]]
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_dc = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_d.copy()
FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_dc["Genre"] = ["Comedy-Drama" if x in list_to_h_c_a_d_r_ab_sff_aa_t_dc else x for x in FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_d["Genre"]]
DataGenre13 = pd.DataFrame(FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_dc['Genre'].value_counts())
DataGenre13 = DataGenre13.reset_index()

In [ ]:
## Dataset with movies classified based on genres listed above (i.e. Horror, Comedy, Animation, Drama, Romance, Biography, Sci-Fi/Fantasy, Action/Adventure, Comedy-Drama)

FinalGenre = FinalCombine_No_NaN_to_h_c_a_d_r_ab_sff_aa_t_dc
print(np.where(FinalGenre["Genre"] == "Western"))
print(np.where(FinalGenre["Genre"] == "Crime, Film-Noir, Mystery"))
print(np.where(FinalGenre["Genre"] == "Comedy, Crime, Mystery"))
FinalGenre.drop([369,454,471,545,522,618,1082,1354], axis=0, inplace=True)
FinalGenrereindex = FinalGenre.reset_index(drop = True)
new_dataset_imdb = FinalGenrereindex

## ISSUE ABOVE

The code above is not runable because indices of observations we classified to be a particular genre for our project will not be the same indices when making a new request (IMDB updates its site with new movies)

In [ ]:
## Stadardizing movie titles

new_dataset_imdb = FinalCombine_No_NaN 
def rare_title_list(list_title_upp, add_apos = True):
    common_character_list = [
    " ",
    "0","1","2","3","4","5","6","7","8","9",
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z"]
    if add_apos == True:
        common_character_list.append("'")
    rare_title_list = []
    for title in list_title_upp:
        condition_title = []
        for character in title:
            condition_title.append(character not in common_character_list)
        if sum(condition_title) >= 1:
            rare_title_list.append(title)
    return rare_title_list
movie_title_imdb = [x.upper()
                    .replace(" : "," ")
                    .replace(": "," ")
                    .replace(" :"," ")
                    .replace(":"," ")
                    .replace(" . "," ")
                    .replace(". "," ")
                    .replace(" ."," ")
                    .replace("."," ")
                    .replace(" - "," ") 
                    .replace(" -"," ")
                    .replace("- "," ")
                    .replace("-"," ")
                    .replace(" \\ ","")
                    .replace("\\ ","")
                    .replace(" \\","")
                    .replace("\\","")
                    .replace("   "," ")
                    .replace(" , "," ")
                    .replace(" ,"," ")
                    .replace(", "," ")
                    .replace(","," ")
                    .replace(" ! "," ")
                    .replace(" !"," ")
                    .replace("! "," ")
                    .replace("!"," ")
                    .replace(" ? "," ")
                    .replace(" ?"," ")
                    .replace("? "," ")
                    .replace("?"," ")
                    .replace(" / "," ")
                    .replace(" /"," ")
                    .replace("/ "," ")
                    .replace("/"," ")
                    .replace(" ( "," ")
                    .replace("( "," ")
                    .replace(" ("," ")
                    .replace("("," ")
                    .replace(" ) "," ")
                    .replace(" )"," ")
                    .replace(") "," ")
                    .replace(")"," ")
                    .replace(" [ "," ")
                    .replace("[ "," ")
                    .replace(" ["," ")
                    .replace("["," ")
                    .replace(" ] "," ")
                    .replace(" ]"," ")
                    .replace("] "," ")
                    .replace("]"," ")
                    .replace(" * "," ")
                    .replace(" *"," ")
                    .replace("* "," ")
                    .replace("*"," ")
                    .replace(" & "," AND ")
                    .replace("& "," AND ")
                    .replace(" &"," AND ")
                    .replace("&"," AND ")
                    .replace("Ä","A")
                    .replace("É","E")
                    .replace("·"," ")
                    .replace("Á","A")             
                   for x in new_dataset_imdb["Title"]]
rare_title_list(movie_title_imdb)
removie_list_imdb = rare_title_list(movie_title_imdb)
movie_title_imdb = [x.rstrip(" ") if x.endswith(" ") else x for x in movie_title_imdb]
movie_title_imdb = [x.lstrip(" ") if x.startswith(" ") else x for x in movie_title_imdb]
clean_movie_title_imbd = pd.DataFrame(movie_title_imdb) #clean movie title
tem_dataset_imdb = pd.concat([new_dataset_imdb,clean_movie_title_imbd],axis=1)
tem_dataset_imdb.columns = [*tem_dataset_imdb.columns[:-1], 'Movie Name']
tem_dataset_imdb = tem_dataset_imdb.loc[[False if x in removie_list_imdb else True for x in tem_dataset_imdb['Movie Name']],:].reset_index(drop=True)
tem_dataset_imdb = tem_dataset_imdb.sort_values(by = 'Movie Name')
tem_dataset_imdb["Title"] = [x + " " + str(y)  for x,y in zip(tem_dataset_imdb['Movie Name'],tem_dataset_imdb["Year"])]
tem_dataset_imdb = tem_dataset_imdb.reset_index(drop=True)
tem_dataset_imdb = tem_dataset_imdb.iloc[:,0:13]
year_list  = tem_dataset_imdb["Year"]
year_list_new = year_list.apply(lambda st: st[st.find("(")+1:st.find(")")])
tem_dataset_imdb["Year"] = year_list_new

In [ ]:
tem_dataset_imdb

## MERGE The Numbers Dataset and IMDB Dataset 

In [ ]:
part_1 = tem_dataset.copy() # The Numbers Dataset
part_2 = tem_dataset_imdb.copy() # IMDB Dataset
part_2.rename({"Title":"Movie"}, axis = "columns", inplace = True)
def final_name_change(text):
    TEXT = (text.
            replace("10 000 B C (2008)","10 000 BC (2008)").
            replace("SOUTHLAND TALES (2007)","SOUTHLAND TALES (2006)").
            replace("SPICE WORLD (1998)","SPICE WORLD (1997)").
            replace("SPIDER MAN INTO THE SPIDER VERSE 3D (2018)","SPIDER MAN INTO THE SPIDER VERSE (2018)").
            replace("SPRING BREAKERS (2013)","SPRING BREAKERS (2012)").
            replace("SPY KIDS 2 ISLAND OF LOST DREAMS (2002)","SPY KIDS 2 THE ISLAND OF LOST DREAMS (2002)").
            replace("SPY KIDS 3 D GAME OVER (2003)","SPY KIDS 3 GAME OVER (2003)").
            replace("SPY KIDS 4 ALL THE TIME IN THE WORLD (2011)","SPY KIDS ALL THE TIME IN THE WORLD (2011)").
            replace("TAE GUIK GI THE BROTHERHOOD OF WAR (2004)","TAE GUK GI THE BROTHERHOOD OF WAR (2004)").
            replace("TEETH (2008)","TEETH (2007)").
            replace("THE ADVENTURES OF SHARKBOY AND LAVAGIRL 3 D (2005)","THE ADVENTURES OF SHARKBOY AND LAVAGIRL IN 3 D (2005)").
            replace("THE BOONDOCK SAINTS (2000)","THE BOONDOCK SAINTS (1999)").
            replace("THE BROWN BUNNY (2004)","THE BROWN BUNNY (2003)").
            replace("THE GREEN INFERNO (2015)","THE GREEN INFERNO (2013)").
            replace("THE HILLS HAVE EYES 2 (2007)","THE HILLS HAVE EYES II (2007)").
            replace("THE INFORMERS (2009)","THE INFORMERS (2008)").
            replace("THE LORDS OF SALEM (2013)","THE LORDS OF SALEM (2012)").
            replace("TWIXT (2012)","TWIXT (2011)").
            replace("WALKING WITH DINOSAURS 3D (2013)","WALKING WITH DINOSAURS (2013)").
            replace("MEMENTO (2001)","MEMENTO (2000)").
            replace("MOVIE 43 (2012)","MOVIE 43 (2013)").
            replace("NIGHT ON EARTH (1992)","NIGHT ON EARTH (1991)").
            replace("OUIJA (II) (2014)","OUIJA (2014)").
            replace("POLICE ACADEMY 7 MISSION TO MOSCOW (1994)","POLICE ACADEMY MISSION TO MOSCOW (1994)").
            replace("PORTRAIT DE LA JEUNE FILLE EN FEU (2019)","PORTRAIT OF A LADY ON FIRE (2019)").
            replace("SHARK NIGHT (2011)","SHARK NIGHT 3D (2011)").
            replace("SILENT HOUSE (2012)","SILENT HOUSE (2011)").
            replace("300 (2007)", "300 (2006)").
            replace("AMORES PERROS (2001)", "AMORES PERROS (2000)").
            replace("AN AMERICAN HAUNTING (2006)", "AN AMERICAN HAUNTING (2005)").
            replace("BLOODRAYNE (2006)","BLOODRAYNE (2005)").
            replace("BOAT TRIP (2003)","BOAT TRIP (2002)").
            replace("CASABLANCA (1943)","CASABLANCA (1942)").
            replace("DARKNESS (2004)","DARKNESS (2002)").
            replace("DAWN OF THE DEAD (1979)","DAWN OF THE DEAD (1978)").
            replace("DOA DEAD OR ALIVE (2007)","DOA DEAD OR ALIVE (2006)").
            replace("DONKEY PUNCH (2009)","DONKEY PUNCH (2008)").
            replace("DYLAN DOG DEAD OF NIGHT (2011)","DYLAN DOG DEAD OF NIGHT (2010)" ).
            replace("EX MACHINA (2015)", "EX MACHINA (2014)").
            replace("EYE OF THE BEHOLDER (2000)", "EYE OF THE BEHOLDER (1999)").
            replace("FATHER'S DAY (1997)", "FATHERS' DAY (1997)").
            replace("FIFTY SHADES FREED (2018)", "FIFTY SHADES DARKER (2017)" ).
            replace("HARRY POTTER AND THE DEATHLY HALLOWS PART 1 (2010)", "HARRY POTTER AND THE DEATHLY HALLOWS PART I (2010)").
            replace("HIGHLANDER THE FINAL DIMENSION (1995)", "HIGHLANDER THE FINAL DIMENSION (1994)").
            replace("IN THE NAME OF THE KING A DUNGEON SIEGE TALE (2008)", "IN THE NAME OF THE KING A DUNGEON SIEGE TALE (2007)").
            replace("JASON X (2002)", "JASON X (2001)").
            replace("KINGSMAN THE SECRET SERVICE (2015)", "KINGSMAN THE SECRET SERVICE (2014)").
            replace("LOCK STOCK AND TWO SMOKING BARRELS (1999)", "LOCK STOCK AND TWO SMOKING BARRELS (1998)"))
    return TEXT
part_1["Movie"]=final_name_change(part_1["Movie"])
part_2["Movie"]=final_name_change(part_2["Movie"])
result_Final=pd.merge(part_1,part_2,on="Movie")
result_final = result_Final.drop("Year",axis=1) 
result_final

## ADDING Oscar Information and Franchise Information

In [ ]:
df = result_final.copy()
df["Year"] = [int(x.split(" (")[1].replace(")","")) for x in df["Movie"]]

In [ ]:
## Extracting Oscar Winners from Oscar Database

url_oscar = 'https://awardsdatabase.oscars.org/search/getresults?query'
query_oscar = {"Sort":"3-Award Category-Chron","AwardShowNumberFrom": 1,"AwardShowNumberTo": 94,"Search": 30,"IsWinnersOnly": "true"}
params_oscar = {'query': json.dumps(query_oscar)} # recasts params to query, see https://stackoverflow.com/questions/42614870/dictionary-type-params-in-query-string
headers_oscar = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36'}
results_oscar = requests.get(url_oscar, headers = headers_oscar, params = params_oscar)
results_oscar.raise_for_status()
html_oscar = lx.fromstring(results_oscar.text)

winners = [string.text_content() 
           if not "\r" in string.text_content() 
           else None 
           for string in html_oscar.xpath("//div[@class = 'awards-result-chron result-group group-awardcategory-chron']//a")]
winners = [x for x in winners if x != None]
all_year = []
one_year = []
for x in winners:
    if (x.endswith("st)") or x.endswith("nd)") or x.endswith("rd)") or x.endswith("th)")) and not (x.endswith("Award)") or x.endswith("and)") or x.endswith("ath)")):
        all_year.append(one_year)
        one_year = [x]
    else: 
        one_year.append(x)
all_year.append(one_year)
all_year = all_year[1:]
oscar_df = pd.DataFrame(columns = ["Year", "Oscar_Actor", "Oscar_Director"])
for i in range(0,len(all_year)):
    df_year = pd.DataFrame(all_year[i]) 
    index_actor_title = df_year.loc[[True if x.startswith("ACT") else False for x in all_year[i]],:].index 
    actor_name = [all_year[i][x+1] for x in index_actor_title] 
    actor_name
    index_director_title = df_year.loc[[True if x.startswith("DIRECTING") else False for x in all_year[i]],:].index #
    director_name = [all_year[i][x+1] if i <= 2 else all_year[i][x+2] for x in index_director_title] 
    director_name
    year_df = pd.DataFrame([{"Year": all_year[i][0], "Oscar_Actor": actor_name, "Oscar_Director": director_name}], index = [i])
    oscar_df = pd.concat([oscar_df, year_df], axis = 0)
oscar_df["Year"] = [x for x in range(1928,2021+1)]
oscar_df

In [ ]:
## Creating columns indicating whether or not a movie has an Oscar lead and Oscar director

def oscar_indicator_row(df, role, year_of_movie, row):
    if role == "Lead":
        oscar_role = "Oscar_Actor"
    elif role == "Director":
        oscar_role = "Oscar_Director"
    list_of_lists = oscar_df.loc[[True if x < year_of_movie else False for x in oscar_df["Year"]],oscar_role]
    one_list = [x for sublist in list_of_lists for x in sublist]
    if type(df[role][row]) != list:
        people_in_row = [df[role][row]]
    else:
        people_in_row = df[role][row]
    condition_list = []
    for i in range(0, len(people_in_row)):
        condition_list.append(people_in_row[i] in one_list)
    if sum(condition_list) >= 1:
        return 1
    else:
        return 0
df["Oscar_Lead"] = [oscar_indicator_row(df, "Lead", year_of_movie, i) for year_of_movie,i in zip(df["Year"],range(0,df.shape[0]))]
df["Oscar_Director"] = [oscar_indicator_row(df, "Director", year_of_movie, i) for year_of_movie,i in zip(df["Year"],range(0,df.shape[0]))]

In [ ]:
## Extracting Top 27 Franchise from Insider 

response_franchise = requests.get("https://www.businessinsider.com/greatest-movie-franchises-all-time-critics-2018-8")
html_franchise = lx.fromstring(response_franchise.text)

list_franchise = [x.text_content().replace("\xa0","").replace('"',"").replace(" (","(").split("(")[0] for x in html_franchise.xpath("//div[@class = 'slide-layout clearfix']//p")]
list_year = [x.text_content().replace("\xa0","").replace('"',"").replace(" (","(").split("(")[1].split(")")[0] for x in html_franchise.xpath("//div[@class = 'slide-layout clearfix']//p")]
list_year = ["(" + x + ")" for x in list_year]
std_list_franchise = [x.upper()
                      .replace(": "," ")
                      .replace(":"," ")
                      .replace(". "," ")
                      .replace("."," ")
                      .replace(" – "," ")
                      .replace(" - "," ")
                      .replace("-", " ")
                      for x in list_franchise]
std_list_franchise_year = [x + " " + year for x,year in zip(std_list_franchise,list_year)]
std_list_franchise_year = [x
                           .replace("HARRY POTTER AND THE DEATHLY HALLOWS PART 1 (2010)","HARRY POTTER AND THE DEATHLY HALLOWS PART I (2010)")
                           .replace("HARRY POTTER AND THE DEATHLY HALLOWS PART 2 (2010)","HARRY POTTER AND THE DEATHLY HALLOWS PART II (2010)")
                           .replace("MARVEL'S THE AVENGERS (2012)", "THE AVENGERS (2012)")
                           for x in std_list_franchise_year]
std_list_franchise_year.append("SPIDER MAN INTO THE SPIDER VERSE (2018)")
std_list_franchise_year.append("STAR TREK (2009)")
std_list_franchise_year.append("STAR TREK II THE WRATH OF KHAN (1982)")
std_list_franchise_year.append("STAR TREK INTO DARKNESS (2013)")
std_list_franchise_year.append("STAR TREK V THE FINAL FRONTIER (1989)")
std_list_franchise_year.append("THE BATMAN (2022)")
std_list_franchise_year

In [ ]:
## Creating column indicating whether or not a movie is part of an established franchise/IP

df["Franchise"] = [1 if x in std_list_franchise_year else 0 for x in df["Movie"]]

## CLEANING Data

In [ ]:
df = df.rename(columns = {"Movie":"movie", 
                          "Production Budget": "budget",
                          "Domestic Gross": "gross_dom",
                          "Worldwide Gross": "gross_wor",
                          "Genre": "genre",
                          "Movie Rating": "rating",
                          "User Rating": "score_user",
                          "Meta Score": "score_meta",
                          "Duration in mins": "mins",
                          "Director": "director",
                          "Lead": "lead",
                          "User Rating Good/Bad": "score_user_good",
                          "Meta Score Good/Bad": "score_meta_good",
                          "Director Count 0/1": "director_one",
                          "Director Count": "director_num",
                          "Year": "year",
                          "Oscar_Lead": "oscar_lead",
                          "Oscar_Director": "oscar_director",
                          "Franchise": "ip"})
df_final_plus = pd.concat([df["year"],
                           df["movie"], 
                           df["gross_dom"],
                           df["gross_wor"],
                           df["budget"],
                           df["score_user"],
                           df["score_user_good"],
                           df["score_meta"],
                           df["score_meta_good"],
                           df["rating"],
                           df["genre"],
                           df["ip"],
                           df["lead"],
                           df["oscar_lead"],
                           df["director"],
                           df["director_num"],
                           df["director_one"],
                           df["oscar_director"],
                           df["mins"]], axis = 1)
df_final = df_final_plus.drop(columns = ["movie", "director", "lead"])
df_final_clean = df_final.copy()
df_final_clean = df_final_clean.loc[[False if x in ["G","Approved","Not Rated","Passed","NC-17","TV-MA","Unrated","GP"] else True for x in df_final["rating"]],:]
df_final_clean["score_user_good"] = [1 if x >= 7 else 0 for x in df_final_clean["score_user"]]
df_final_clean["log_year"] = np.log(df_final_clean["year"])
df_final_clean["log_gross_wor"] = np.log(df_final_clean["gross_wor"])
df_final_clean["log_budget"] = np.log(df_final_clean["budget"])
df_final_clean["log_mins"] = np.log(df_final_clean["mins"])
df_final_clean = df_final_clean.drop(columns = ["gross_dom", "director_num", "score_meta_good"])
df_final_clean = df_final_clean.reset_index(drop = True)
df_final_clean_dummies = pd.get_dummies(df_final_clean)
df_final_clean_dummies = df_final_clean_dummies.rename(columns = {"rating_PG": "rating_pg", 
                                                                  "rating_PG-13": "rating_pg_13",
                                                                  "rating_R": "rating_r",
                                                                  "genre_Action/Adventure": "genre_action_adv",
                                                                  "genre_Animation": "genre_animation",
                                                                  "genre_Biography": "genre_bio",
                                                                  "genre_Comedy": "genre_comedy",
                                                                  "genre_Comedy-Drama": "genre_comedy_drama",
                                                                  "genre_Drama": "genre_drama",
                                                                  "genre_Horror": "genre_horror",
                                                                  "genre_Romance": "genre_romance",
                                                                  "genre_Sci-fi/Fantasy": "genre_fantasy_sci"})

## Final Dataset for Analysis

IMPORTANT: WE CANNOT SHOW GENRE-RELATED COLUMNS BECAUSE OF REINDEXING ISSUE EXPLAINED EARLIER

IMPORTANT: SLIGHTLY DIFFERENT DATA (I.E. IT CONTAINS 2 LESS OBSERVATIONS THAN IN OUR PROJECT)

In [ ]:
col_names = df_final_clean_dummies.columns
df_final_clean_dummies = df_final_clean_dummies.loc[:,["genre" not in x for x in col_names]]
df_final_clean_dummies

## Modules for Visualization and Tables

In [ ]:
import re
import json
import requests
import requests_cache
requests_cache.install_cache("STA_141B_Final_Project")
import time
import lxml.html as lx 
import numpy as np
import pandas as pd
import seaborn as sns
import plotnine as p9
import statsmodels.api as sm # For Linear Regression ANOVA (WARNING: str() ISSUES) 
import statsmodels.stats.api as sms # For Linear Regression BP test
import statsmodels.formula.api as smf # For Linear Regression to have specific formula structure for model reduction
from statsmodels.stats.outliers_influence import variance_inflation_factor # For VIF
import matplotlib.pyplot as plt # For Linear Regression Assumption Graphs
from matplotlib import gridspec # For Linear Regression Assumption Graphs
from IPython.display import display_html # For Linear Regression Assumption Graphs
from IPython.display import display, HTML # For Side-By-Side Pandas Dataframe
from itertools import chain,cycle # For Linear Regression Assumption Graphs
import scipy # For Logistic Regression LR test 
from scipy.stats import norm # For Linear Regression QQ-plot
import plotly.express as px # For Linear Regression Assumption Graphs
import plotly.graph_objects as go # For Linear Regression Assumption Graphs
import plotly.figure_factory as ff # For Distribution Graph
import plotly.io as pio # For Interactive Graph
pio.renderers.default='notebook' # In order for plotly plots to show
import warnings 
warnings.filterwarnings('ignore')

## Functions for Visualization and Tables

In [ ]:
def highlight_problem_rows(s):
    return 'background-color: %s' % 'red'
def display_side_by_side(dfs:list, captions:list):
    output = ""
    combined = dict(zip(captions, dfs))
    styles = [dict(selector="caption", props=[("font-size", "150%"),("color", "black")])]
    for caption, df in combined.items():
        output += df.set_table_attributes("style='display:inline'").set_caption(caption).set_table_styles(styles)._repr_html_()
        output += "\xa0\xa0\xa0"
    display(HTML(output))
def lin_assump(f, df, res = False, qqp = False, lev = False, coef = False, vif = False, tab = False):
    ols_fit = smf.ols(formula = f, data = df).fit() 
    results_as_html1 = ols_fit.summary().tables[1].as_html()
    coef_pval_df = pd.read_html(results_as_html1, header=0, index_col=0)[0].iloc[:,:-2]
    insig_rows = [x for x,y in zip(coef_pval_df.index,coef_pval_df["P>|t|"]) if y > 0.05]
    coef_pval_df_highlight = coef_pval_df.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows,:])
    outliers_df = pd.DataFrame({"abs_res": abs(ols_fit.resid_pearson)},index = df.index)
    outliers_df = outliers_df[outliers_df["abs_res"] > 3].sort_values(by=["abs_res"], ascending = False)
    outliers_df_highlight = outliers_df.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[:,:])
    results_as_html2 = ols_fit.summary().tables[2].as_html()
    dw_stat = round(float(pd.read_html(results_as_html2, header=0, index_col=0)[0].columns[2]), 5)
    if dw_stat > 1.5 and dw_stat < 2.5:
        dw_conclude = "NOT CORRELATED"
    else:
        dw_conclude = "PROBLEM: CORRELATED"
    dw_row = [dw_stat,dw_conclude]
    results_as_html0 = ols_fit.summary().tables[0].as_html()
    r_sq_adj = pd.read_html(results_as_html0, header=0, index_col=0)[0].iloc[0:6,1:].iloc[[0,4,5],1][0]
    if r_sq_adj >= 0.5:
        r_sq_adj_conclude = "MODEL EXPLAIN AT LEAST HALF VAR"
    else:
        r_sq_adj_conclude = "PROBLEM: MODEL EXPLAIN LESS THAN HALF VAR"
    r_sq_adj_row = [r_sq_adj,r_sq_adj_conclude]
    stat_df = pd.DataFrame([dw_row,r_sq_adj_row], index = ["error_corr_dw","r_sq_adj"], columns = ["stat", "conclude"])
    problem_rows = [x for x,y in zip(stat_df.index,stat_df["conclude"]) if "PROBLEM" in y]
    stat_df_highlight = stat_df.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[problem_rows,:])
    output_var = f.split(" ~")[0]
    input_var = f.replace(output_var + " ~ ","").split(" + ")
    df_in = df[input_var]
    df_in_dummies = pd.get_dummies(df_in)
    vif_data = pd.DataFrame()
    vif_data["feature"] = df_in_dummies.columns
    vif_data["VIF"] = [variance_inflation_factor(df_in_dummies.values, i) for i in range(len(df_in_dummies.columns))]
    vif_data = vif_data.set_index("feature")
    vif_data.index.name = None
    vif_data = vif_data.sort_values(by = "VIF", ascending = False)
    VIF_problem_rows = [x for x,y in zip(vif_data.index,vif_data["VIF"]) if y > 5]
    vif_data_highlight = vif_data.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[VIF_problem_rows,:])
    fit_res_plot = (p9.ggplot(df,mapping = p9.aes(x = 'ols_fit.fittedvalues', y = 'ols_fit.resid_pearson'))
        + p9.geom_point()
        + p9.labs(title = "Fitted vs Residuals", x = 'Fitted values', y = 'Residuals')
        + p9.geom_text(p9.aes(label = [x if abs(y) > 3 else "" for x,y in zip(df.index, ols_fit.resid_pearson)]), color = 'red', size = 10)
        + p9.theme_bw())
    res_quant_df = pd.DataFrame({"res": ols_fit.resid_pearson}, index = df.index)
    res_quant_df = res_quant_df.sort_values(by = ["res"])
    fi = [(i - 0.5) / len(ols_fit.resid_pearson) for i in range(1,len(ols_fit.resid_pearson)+1)] # percentile
    res_quant_df["quant"] = [norm.ppf(x) for x in fi]
    QQ_plot = (p9.ggplot(res_quant_df,mapping = p9.aes(x = "quant", y = "res"))
        + p9.geom_point()
        + p9.geom_abline(p9.aes(intercept = 0, slope = 1), color = 'blue')
        + p9.labs(title = "QQ-Plot with Pearson Residuals")  
        + p9.geom_text(p9.aes(label = [x if abs(y) > 3 else "" for x,y in zip(res_quant_df.index, res_quant_df["res"])]), color = 'red', size = 10)
        + p9.theme_bw())  
    lev_res_plot = (p9.ggplot(df,mapping = p9.aes(x = 'ols_fit.get_influence().hat_matrix_diag', y = 'ols_fit.resid_pearson'))
        + p9.geom_point()
        + p9.labs(title = "Leverage vs Residuals", x = 'Leverage', y = 'Residuals')
        + p9.geom_text(p9.aes(label = [x if abs(y) > 3 else "" for x,y in zip(df.index, ols_fit.resid_pearson)]), color = 'red', size = 10)
        + p9.theme_bw()) 
    if res == True:
        res_df = pd.DataFrame(index = df.index)
        res_df["fit_val"] = ols_fit.fittedvalues
        res_df["res"] = ols_fit.resid_pearson
        fit_res_fig = px.scatter(res_df, x = "fit_val", y = "res",
                                 labels = {"fit_val": "Fitted Values",
                                           "res": "Residuals"},
                                 title = "Fitted vs Residuals",
                                 color_discrete_sequence = ["black"])
        fit_res_fig.add_hline(y = 3, line_width = 1.5, line_dash = "dash", line_color = "red", annotation_font_color = "red", annotation_text = "Outlier Limit", annotation_position="bottom right")
        fit_res_fig.add_hrect(y0 = 3, y1 = 100, fillcolor="red", opacity=0.1, line_width=0)
        fit_res_fig.add_hline(y = -3, line_width = 1.5, line_dash = "dash", line_color = "red", annotation_font_color = "red", annotation_text = "Outlier Limit", annotation_position="top right")
        fit_res_fig.add_hrect(y0 = -100, y1 = -3, fillcolor="red", opacity=0.1, line_width=0)
        fit_res_fig.add_trace(go.Scatter(
            x = [x for x,y in zip(res_df["fit_val"], res_df["res"]) if abs(y) > 3],
            y = [y for y in res_df["res"] if abs(y) > 3],
            mode = "markers+text",
            name = "Outlier",
            text = [x for x,y in zip(res_df.index, res_df["res"]) if abs(y) > 3],
            textfont={"color":"red"},
            textposition = "bottom center",
            marker = {"color": "red"}))
        fit_res_fig.update_xaxes(showgrid=True, gridwidth=1.5, gridcolor='white')
        fit_res_fig.update_yaxes(showgrid=True, gridwidth=1.5, gridcolor='white', range=[-6,6])
        fit_res_fig.update_layout({
            'plot_bgcolor': 'rgb(235,235,235)'})
        fit_res_fig.show()
    elif qqp == True:
        res_quant_df = pd.DataFrame(index = df.index)
        res_quant_df["res"] = ols_fit.resid_pearson
        res_quant_df = res_quant_df.sort_values(by = ["res"])
        fi = [(i - 0.5) / len(ols_fit.resid_pearson) for i in range(1,len(ols_fit.resid_pearson)+1)] # percentile
        res_quant_df["quant"] = [norm.ppf(x) for x in fi]
        res_quant_fig = px.scatter(res_quant_df, x = "quant", y = "res",
                                   labels = {"quant": "Theoretical Quantiles",
                                             "res": "Sample Quantiles"},
                                   title = "QQ-Plot with Pearson Residuals",
                                   color_discrete_sequence = ["black"])
        res_quant_fig.add_trace(go.Scatter(
            x = [-100,100],
            y = [-100,100],
            name = "Normality Line",
            line_color = "blue"))
        res_quant_fig.add_trace(go.Scatter(
            x = [x for x,y in zip(res_quant_df["quant"], res_quant_df["res"]) if abs(y) > 3],
            y = [y for y in res_quant_df["res"] if abs(y) > 3],
            mode = "markers+text",
            name = "Outlier",
            text = [x for x,y in zip(res_quant_df.index, res_quant_df["res"]) if abs(y) > 3],
            textfont={"color":"red"},
            textposition = "bottom center",
            marker = {"color": "red"}))
        res_quant_fig.update_xaxes(showgrid=True, gridwidth=1.5, gridcolor='white', range=[-6, 6])
        res_quant_fig.update_yaxes(showgrid=True, gridwidth=1.5, gridcolor='white', range=[-6, 6])
        res_quant_fig.update_layout({
            'plot_bgcolor': 'rgb(235,235,235)'})
        res_quant_fig.show()
    elif lev == True:
        lev_df = pd.DataFrame(index = df.index)
        lev_df["lev"] = ols_fit.get_influence().hat_matrix_diag
        lev_df["res"] = ols_fit.resid_pearson
        lev_res_fig = px.scatter(lev_df, x = "lev", y = "res",
                                 labels = {"lev": "Leverage",
                                           "res": "Residuals"},
                                 title = "Leverage vs Residuals",
                                 color_discrete_sequence = ["black"])
        lev_res_fig.add_hline(y = 3, line_width = 1.5, line_dash = "dash", line_color = "red", annotation_font_color = "red", annotation_text = "Outlier Limit", annotation_position="bottom right")
        lev_res_fig.add_hrect(y0 = 3, y1 = 100, fillcolor="red", opacity=0.1, line_width=0)
        lev_res_fig.add_hline(y = -3, line_width = 1.5, line_dash = "dash", line_color = "red", annotation_font_color = "red", annotation_text = "Outlier Limit", annotation_position="top right")
        lev_res_fig.add_hrect(y0 = -100, y1 = -3, fillcolor="red", opacity=0.1, line_width=0)
        lev_res_fig.add_trace(go.Scatter(
            x = [x for x,y in zip(lev_df["lev"], lev_df["res"]) if abs(y) > 3],
            y = [y for y in lev_df["res"] if abs(y) > 3],
            mode = "markers+text",
            name = "Outlier",
            text = [x for x,y in zip(lev_df.index, lev_df["res"]) if abs(y) > 3],
            textfont={"color":"red"},
            textposition = "bottom center",
            marker = {"color": "red"}))
        lev_res_fig.update_xaxes(showgrid=True, gridwidth=1.5, gridcolor='white')
        lev_res_fig.update_yaxes(showgrid=True, gridwidth=1.5, gridcolor='white', range=[-6,6])
        lev_res_fig.update_layout({
            'plot_bgcolor': 'rgb(235,235,235)'})
        lev_res_fig.show()
    elif coef == True:
        display(coef_pval_df_highlight)
    elif vif == True:
        display(vif_data_highlight)
    elif tab == True:
        coef_pval_df_core = pd.read_html(results_as_html1, header=0, index_col=0)[0].iloc[:,:-2][["coef", "std err", "P>|t|"]]
        coef_pval_df_core_highlight = coef_pval_df_core.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows,:])
        p1 = fit_res_plot
        p2 = QQ_plot 
        p3 = lev_res_plot 
        fig = (p9.ggplot()+p9.geom_blank(data=df)+p9.theme_void()+p9.theme(figure_size=(18, 4.5))).draw() 
        gs = gridspec.GridSpec(nrows = 1, ncols = 3)
        ax1 = fig.add_subplot(gs[0,0])
        ax2 = fig.add_subplot(gs[0,1])
        ax3 = fig.add_subplot(gs[0,2])
        ax1.title.set_text('Fitted vs Residuals')
        ax2.title.set_text('QQ-Plot with Pearson Residuals')
        ax3.title.set_text('Leverage vs Residuals')
        p1._draw_using_figure(fig, [ax1])
        p2._draw_using_figure(fig, [ax2])
        p3._draw_using_figure(fig, [ax3])
        plt.show()
        display_side_by_side([coef_pval_df_core_highlight, outliers_df_highlight, stat_df_highlight], ["Coefficients", "Outliers", "Other Stats"])
    else:
        coef_pval_df_core = pd.read_html(results_as_html1, header=0, index_col=0)[0].iloc[:,:-2][["coef", "std err", "P>|t|"]]
        coef_pval_df_core_highlight = coef_pval_df_core.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows,:])
        display_side_by_side([coef_pval_df_core_highlight, outliers_df_highlight, stat_df_highlight], ["Coefficients", "Outliers", "Other Stats"])   
def model_red(df, model = "linear", outvar = None, invar = None, f = None, rm = None):
    if model == "linear":
        ols_fit = smf.ols(f, data = df).fit() 
        results_as_html1 = ols_fit.summary().tables[1].as_html()
        coef_pval_df = pd.read_html(results_as_html1, header=0, index_col=0)[0].iloc[:,:-2]
        insig_rows = [x for x,y in zip(coef_pval_df.index,coef_pval_df["P>|t|"]) if y > 0.05]
        coef_pval_df_highlight = coef_pval_df.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows,:])
        output_var = f.split(" ~")[0]
        input_var = f.replace(output_var + " ~ ","").split(" + ")
        df_in = df[input_var]
        df_in_dummies = pd.get_dummies(df_in)
        vif_data = pd.DataFrame()
        vif_data["feature"] = df_in_dummies.columns
        vif_data["VIF"] = [variance_inflation_factor(df_in_dummies.values, i) for i in range(len(df_in_dummies.columns))]
        vif_data = vif_data.set_index("feature")
        vif_data.index.name = None
        vif_data = vif_data.sort_values(by = "VIF", ascending = False)
        VIF_problem_rows = [x for x,y in zip(vif_data.index,vif_data["VIF"]) if y > 5]
        vif_data_highlight = vif_data.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[VIF_problem_rows,:])
        results_as_html0 = ols_fit.summary().tables[0].as_html()
        aic_bic = pd.read_html(results_as_html0, header=0, index_col=0)[0].iloc[[4,5],2]
        aic_bic = [x for x in aic_bic]
        aic_bic_df = pd.DataFrame(index = ["AIC", "BIC"])
        aic_bic_df["stat"] = aic_bic
        if rm == None:
            display_side_by_side([coef_pval_df_highlight, vif_data_highlight], ["Coefficients", "VIF"])
        else:
            remove_var_minus = ["-" + x for x in rm]
            string = ""
            for x in remove_var_minus:
                string += x
            ols_fit_red = smf.ols(f + string, data = df).fit() 
            results_as_html1_red = ols_fit_red.summary().tables[1].as_html()
            coef_pval_df_red = pd.read_html(results_as_html1_red, header=0, index_col=0)[0].iloc[:,:-2]
            insig_rows_red = [x for x,y in zip(coef_pval_df_red.index,coef_pval_df_red["P>|t|"]) if y > 0.05]
            coef_pval_df_highlight_red = coef_pval_df_red.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows_red,:])    
            input_var_red = input_var.copy() 
            for i in rm:
                input_var_red.remove(i)
            df_in_red = df[input_var_red]
            df_in_dummies_red = pd.get_dummies(df_in_red)
            vif_data_red = pd.DataFrame()
            vif_data_red["feature"] = df_in_dummies_red.columns
            vif_data_red["VIF"] = [variance_inflation_factor(df_in_dummies_red.values, i) for i in range(len(df_in_dummies_red.columns))]
            vif_data_red = vif_data_red.set_index("feature")
            vif_data_red.index.name = None
            vif_data_red = vif_data_red.sort_values(by = "VIF", ascending = False)
            VIF_problem_rows_red = [x for x,y in zip(vif_data_red.index,vif_data_red["VIF"]) if y > 5]
            vif_data_highlight_red = vif_data_red.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[VIF_problem_rows_red,:])
            results_as_html0_red = ols_fit_red.summary().tables[0].as_html()
            aic_bic_red = pd.read_html(results_as_html0_red, header=0, index_col=0)[0].iloc[[4,5],2]
            aic_bic_red = [x for x in aic_bic_red]
            aic_bic_df_red = pd.DataFrame(index = ["AIC", "BIC"])
            aic_bic_df_red["stat"] = aic_bic_red
            display_side_by_side([coef_pval_df_highlight, vif_data_highlight, coef_pval_df_highlight_red, vif_data_highlight_red], ["Coefficients", "VIF", "Coefficients of Reduced", "VIF of Reduced"])  
    elif model == "logit":
        df_logit = sm.add_constant(df)
        const_invar = ["const"]
        for i in invar:
            const_invar.append(i)
        logit_fit = sm.GLM(df_logit[outvar], df_logit[const_invar], family=sm.families.Binomial()).fit()
        coef_pval_df_logit = pd.read_html(logit_fit.summary().tables[1].as_html(), header=0, index_col=0)[0].iloc[:,:-2]
        insig_rows_logit = [x for x,y in zip(coef_pval_df_logit.index,coef_pval_df_logit["P>|z|"]) if y > 0.05]
        coef_pval_df_highlight_logit = coef_pval_df_logit.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows_logit,:])
        df_in = df[invar]
        df_in_dummies = pd.get_dummies(df_in)
        vif_data = pd.DataFrame()
        vif_data["feature"] = df_in_dummies.columns
        vif_data["VIF"] = [variance_inflation_factor(df_in_dummies.values, i) for i in range(len(df_in_dummies.columns))]
        vif_data = vif_data.set_index("feature")
        vif_data.index.name = None
        vif_data = vif_data.sort_values(by = "VIF", ascending = False)
        VIF_problem_rows = [x for x,y in zip(vif_data.index,vif_data["VIF"]) if y > 5]
        vif_data_highlight = vif_data.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[VIF_problem_rows,:])
        if rm == None:
            display_side_by_side([coef_pval_df_highlight_logit, vif_data_highlight], ["Coefficients", "VIF"])
        else:
            const_invar_red = const_invar.copy()
            for i in rm:
                const_invar_red.remove(i)
            logit_fit_red = sm.GLM(df_logit[outvar], df_logit[const_invar_red], family=sm.families.Binomial()).fit()
            coef_pval_df_logit_red = pd.read_html(logit_fit_red.summary().tables[1].as_html(), header=0, index_col=0)[0].iloc[:,:-2]
            insig_rows_logit_red = [x for x,y in zip(coef_pval_df_logit_red.index,coef_pval_df_logit_red["P>|z|"]) if y > 0.05]
            coef_pval_df_highlight_logit_red = coef_pval_df_logit_red.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[insig_rows_logit_red,:])
            invar_red = invar.copy()
            for i in rm:
                invar_red.remove(i)
            df_in_red = df[invar_red]
            df_in_dummies_red = pd.get_dummies(df_in_red)
            vif_data_red = pd.DataFrame()
            vif_data_red["feature"] = df_in_dummies_red.columns
            vif_data_red["VIF"] = [variance_inflation_factor(df_in_dummies_red.values, i) for i in range(len(df_in_dummies_red.columns))]
            vif_data_red = vif_data_red.set_index("feature")
            vif_data_red.index.name = None
            vif_data_red = vif_data_red.sort_values(by = "VIF", ascending = False)
            VIF_problem_rows_red = [x for x,y in zip(vif_data_red.index,vif_data_red["VIF"]) if y > 5]
            vif_data_highlight_red = vif_data_red.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[VIF_problem_rows_red,:])
            display_side_by_side([coef_pval_df_highlight_logit, vif_data_highlight, coef_pval_df_highlight_logit_red, vif_data_highlight_red], ["Coefficients", "VIF", "Coefficients of Reduced", "VIF of Reduced"])

## User Score Visualization and Tables

IMPORTANT: MAY DIFFER FROM PROJECT REPORT GIVEN THE SLIGHTLY DIFFERENT DATA PRODUCED ABOVE

In [ ]:
quant_var = ["year", "gross_wor", "budget", "mins", "score_meta"]
fig = (p9.ggplot()+p9.geom_blank(data=df_final_clean_dummies)+p9.theme_void()+p9.theme(figure_size=(24, 4.5))).draw() 
gs = gridspec.GridSpec(nrows = 1, ncols = 5)
for i,j in zip(quant_var,range(0,5)):
    plot = (
        p9.ggplot(df_final_clean_dummies,p9.aes(x = i, y = "score_user"))
        + p9.geom_point()
    )
    ax = fig.add_subplot(gs[0,j])
    ax.title.set_text(i + " vs score_user")
    plot._draw_using_figure(fig, [ax])
plt.show()

In [ ]:
fig = ff.create_distplot([df_final_clean_dummies["score_user"]], group_labels = ["score_user Density"], colors = ["black"], show_hist = False, show_rug = False)
fig.update_layout({'plot_bgcolor': 'rgb(235,235,235)'}, showlegend = False, title = "score_user Distribution")
fig.show()

In [ ]:
log_quant_var = ["log_year", "log_gross_wor", "log_budget", "log_mins", "score_meta"]
fig = (p9.ggplot()+p9.geom_blank(data=df_final_clean_dummies)+p9.theme_void()+p9.theme(figure_size=(24, 4.5))).draw() 
gs = gridspec.GridSpec(nrows = 1, ncols = 5)
for i,j in zip(log_quant_var,range(0,5)):
    plot = (
        p9.ggplot(p9.aes(x = i))
        + p9.geom_density(df_final_clean_dummies[df_final_clean_dummies["score_user_good"] == 1], color = "green", fill = "green", alpha = .1)
        + p9.geom_density(df_final_clean_dummies[df_final_clean_dummies["score_user_good"] == 0], color = "red", fill = "red", alpha = .1)
    )
    ax = fig.add_subplot(gs[0,j])
    ax.title.set_text(i + " Distribution")
    plot._draw_using_figure(fig, [ax])
plt.show()

## ISSUE BELOW

The code below is not runable because of the absense of genre-related columns

In [ ]:
def conditional_prob_df(list_of_binary_input, df):
    combine_df = pd.DataFrame(index = list_of_binary_input, columns = ["P(score_user_good=1|x=1)","P(score_user_good=1|x=0)"]) 
    for i in list_of_binary_input:
        input_df = pd.crosstab(df["score_user_good"],df[i])
        prob1_given1 = input_df.iloc[1,1]/sum(input_df.iloc[:,1])
        prob1_given0 = input_df.iloc[1,0]/sum(input_df.iloc[:,0])
        combine_df.loc[i,:] = [prob1_given1,prob1_given0]
    diff_prob = abs(combine_df.iloc[:,1] - combine_df.iloc[:,0])
    problem_rows = [x for x,y in zip(list_of_binary_input,diff_prob) if y < 0.10]
    combine_df_highlight = combine_df.style.applymap(highlight_problem_rows, subset=pd.IndexSlice[problem_rows,:])
    display_side_by_side([combine_df_highlight], ["Probability of Good User Score Given X"])
conditional_prob_df(['ip',
                     'oscar_lead',
                     'director_one',
                     'oscar_director',
                     'rating_pg',
                     'rating_pg_13',
                     'rating_r',
                     'genre_action_adv',
                     'genre_animation',
                     'genre_bio', 
                     'genre_comedy',
                     'genre_comedy_drama', 
                     'genre_drama', 
                     'genre_horror', 
                     'genre_romance',
                     'genre_fantasy_sci'], df_final_clean_dummies)

## ISSUE ABOVE

The code above is not runable because of the absense of genre-related columns

In [ ]:
## These are NOT the exact arguments we inputted for our project because of the absense of genre-related columns
## Instead, this is an example to show how the model_red() function works

model_red(df = df_final_clean_dummies, model = "logit", outvar = ["score_user_good"], invar = ["log_year", 
                                                                                               "log_budget",
                                                                                               "log_gross_wor",
                                                                                               "log_mins",
                                                                                               "score_meta",
                                                                                               "ip",
                                                                                               "oscar_lead",
                                                                                               "director_one",
                                                                                               "oscar_director",
                                                                                               "rating_pg",
                                                                                               "rating_pg_13",
                                                                                               "rating_r"], rm = ["rating_r"])

## World Gross Visualization and Tables

IMPORTANT: MAY DIFFER FROM PROJECT REPORT GIVEN THE SLIGHTLY DIFFERENT DATA PRODUCED ABOVE

In [ ]:
quant_var = ["log_year", "log_budget", "log_mins", "score_meta"]
fig = (p9.ggplot()+p9.geom_blank(data=df_final_clean_dummies)+p9.theme_void()+p9.theme(figure_size=(22, 4.5))).draw() 
gs = gridspec.GridSpec(nrows = 1, ncols = 4)
for i,j in zip(quant_var,range(0,4)):
    plot = (
        p9.ggplot(df_final_clean_dummies,p9.aes(x = i, y ="log_gross_wor"))
        + p9.geom_point()
        + p9.theme_bw()
    )
    ax = fig.add_subplot(gs[0,j])
    ax.title.set_text(i + " vs log_gross_wor")
    plot._draw_using_figure(fig, [ax])
plt.show()

In [ ]:
sns.set(rc={'figure.figsize':(25,10)})
f, axes = plt.subplots(1, 5)
sns.boxplot(y="log_gross_wor", x= "score_user_good", data=df_final_clean_dummies,  orient='v' , ax=axes[0])
sns.boxplot(y="log_gross_wor", x= "ip", data=df_final_clean_dummies,  orient='v' , ax=axes[1])
sns.boxplot(y="log_gross_wor", x= "oscar_lead", data=df_final_clean_dummies,  orient='v' , ax=axes[2])
sns.boxplot(y="log_gross_wor", x= "director_one", data=df_final_clean_dummies,  orient='v' , ax=axes[3])
sns.boxplot(y="log_gross_wor", x= "oscar_director", data=df_final_clean_dummies,  orient='v' , ax=axes[4])
plt.show()

In [ ]:
df_rating = df_final_clean_dummies[["log_gross_wor","rating_pg","rating_pg_13","rating_r"]]
df_rating ["rating"] = ["pg" if x==1 
                        else "pg_13" if y == 1 
                        else "r" 
                        for x,y,z in zip(df_rating["rating_pg"],df_rating["rating_pg_13"],df_rating["rating_r"])] 
(
    p9.ggplot(df_rating,p9.aes(x="rating",y="log_gross_wor",color="rating"))
    + p9.geom_boxplot(size = 1) 
    + p9.theme(figure_size=(16, 8))
).draw();

## ISSUE BELOW

The code below is not runable because of the absense of genre-related columns

In [ ]:
df_genre = df_final_clean_dummies[["log_gross_wor","genre_action_adv","genre_animation","genre_bio","genre_comedy","genre_comedy_drama","genre_drama","genre_horror","genre_romance","genre_fantasy_sci"]]
genre_list = [df_genre.iloc[:,i] for i in range(1,10)]
genre_type = ['action_adv','animation','bio','comedy','comedy_drama','drama','horror','romance','fantasy_sci']
genre = pd.DataFrame([None for i in range(len(df_genre))])
for i,j in zip(genre_list,genre_type):
    idx = i.index[i == 1]
    genre.iloc[idx,:] = j
df_genre["genre"] = genre.iloc[:,0]
(
    p9.ggplot(df_genre)  # What data to use
    + p9.aes(x="genre", y="log_gross_wor",color="genre")  
    + p9.geom_boxplot(size = 1)
    + p9.theme(figure_size=(16, 8))
).draw();

## ISSUE ABOVE

The code above is not runable because of the absense of genre-related columns

In [ ]:
## These are NOT the exact arguments we inputted for our project because of the absense of genre-related columns
## Instead, this is an example to show how the lin_assump() function works

lin_assump("log_gross_wor ~ log_year + log_budget + log_mins + score_meta + score_user_good + ip + oscar_lead + director_one + oscar_director + rating_pg + rating_pg_13 + rating_r", df_final_clean_dummies, res = True)

In [ ]:
## These are NOT the exact arguments we inputted for our project because of the absense of genre-related columns
## Instead, this is an example to show how the lin_assump() function works

lin_assump("log_gross_wor ~ log_year + log_budget + log_mins + score_meta + score_user_good + ip + oscar_lead + director_one + oscar_director + rating_pg + rating_pg_13 + rating_r", df_final_clean_dummies, qqp = True)

In [ ]:
## These are NOT the exact arguments we inputted for our project because of the absense of genre-related columns
## Instead, this is an example to show how the lin_assump() function works

lin_assump("log_gross_wor ~ log_year + log_budget + log_mins + score_meta + score_user_good + ip + oscar_lead + director_one + oscar_director + rating_pg + rating_pg_13 + rating_r", df_final_clean_dummies, lev = True)

In [ ]:
## These are NOT the exact arguments we inputted for our project because of the absense of genre-related columns
## Instead, this is an example to show how the lin_assump() function works

lin_assump("log_gross_wor ~ log_year + log_budget + log_mins + score_meta + score_user_good + ip + oscar_lead + director_one + oscar_director + rating_pg + rating_pg_13 + rating_r", df_final_clean_dummies)

In [ ]:
## These are NOT the exact arguments we inputted for our project because of the absense of genre-related columns
## Instead, this is an example to show how the model_red() function works

model_red(f = "log_gross_wor ~ log_year + log_budget + log_mins + score_meta + score_user_good + ip + oscar_lead + director_one + oscar_director + rating_pg + rating_pg_13 + rating_r", df = df_final_clean_dummies, rm = ["rating_r"])